In [9]:
import pandas as pd
from io import StringIO

def expand_table_with_missing_bpm(df):
    # Skip the first two rows (header and units)
    data = df.iloc[1:].copy()
    
    # Convert columns to numeric where applicable
    data['HR'] = pd.to_numeric(data['HR'], errors='coerce')
    data['Calories'] = pd.to_numeric(data['Calories'], errors='coerce')
    
    # Create a list to store expanded rows
    expanded_rows = []
    
    # Iterate through rows to interpolate missing BPM values
    for i in range(len(data) - 1):
        current_row = data.iloc[i]
        next_row = data.iloc[i + 1]
        
        current_bpm = current_row['HR']
        next_bpm = next_row['HR']
        
        # Add the current row to the expanded rows
        # Calculate calories per second
        current_row_copy = current_row.copy()
        current_row_copy['Calories'] = current_row_copy['Calories'] / 60 
        expanded_rows.append(current_row_copy)
        
        # Check if there are missing BPM values
        if next_bpm - current_bpm > 1:
            missing_bpm_count = int(next_bpm - current_bpm - 1)
            calorie_diff = (next_row['Calories'] - current_row['Calories']) / (missing_bpm_count + 1)
            
            # Generate missing rows
            for j in range(1, missing_bpm_count + 1):
                interpolated_bpm = current_bpm + j
                interpolated_calories = current_row['Calories'] + calorie_diff * j
                
                # Create a new row with interpolated values
                interpolated_row = current_row.copy()
                interpolated_row['HR'] = interpolated_bpm
                # Calculate calories per second based on the interpolated value
                interpolated_row['Calories'] = interpolated_calories / 60
                
                # Add the interpolated row to the expanded rows
                expanded_rows.append(interpolated_row)
    
    # Add the last row to the expanded rows
    last_row = data.iloc[-1].copy()
    # Calculate calories per second
    last_row['Calories'] = last_row['Calories'] / 60 
    expanded_rows.append(last_row)
    
    # Convert the list of rows back to a DataFrame
    expanded_data = pd.DataFrame(expanded_rows)

    # Add a new column for calories per hour
    expanded_data['Calories / Min'] = expanded_data['Calories'] * 60
    
    return expanded_data

In [15]:
import pandas as pd
from io import StringIO

# Example usage
csv_data = """Time	HR	VO2	VO2	VE/VO2	VCO2	VE/VCO2	RER	Calories	Fat	CHO
min:sec	BPM	mL/min	mL/kg/min		mL/kg/min			Cals/min	%	%
00:00	96	478	5.94	33.46	5.08	39.18	0.856	2.33	47.4	52.6
00:14	106	788	9.79	31.34	8.83	34.72	0.902	3.88	31.7	68.3
00:16	109	798	10.79	31.34	8.83	34.72	0.902	4.34	31.7	68.3"""

# Read the CSV data into a DataFrame
# df = pd.read_csv(StringIO(csv_data), sep="\t")
df = pd.read_csv("./v02max_data.csv")
#print(df)

# Keep only HR and Calories columns
df = df[['HR', 'Calories']]

# Expand the table
expanded_df = expand_table_with_missing_bpm(df)

# Set display options to show all rows and columns without truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.6f}'.format)  # Optional: for better formatting of float values

# Display the expanded DataFrame
display(expanded_df)

,HR,Calories,Calories / Min
1,96.000000,0.038833,2.330000
1,97.000000,0.041417,2.485000
1,98.000000,0.044000,2.640000
1,99.000000,0.046583,2.795000
1,100.000000,0.049167,2.950000
1,101.000000,0.051750,3.105000
1,102.000000,0.054333,3.260000
1,103.000000,0.056917,3.415000
1,104.000000,0.059500,3.570000
1,105.000000,0.062083,3.725000
